# MiniCV — Demonstration Notebook

This notebook demonstrates the core functionality of the MiniCV library.

**CSE480: Machine Vision | Milestone 1 | Spring 2026**

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import minicv

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100

## 1. Create a Sample Image

Since we cannot use external images, we create a synthetic test image.

In [ ]:
# Create a synthetic grayscale test image with gradients and shapes
h, w = 128, 128
img = np.zeros((h, w), dtype=np.uint8)

# Horizontal gradient
for col in range(w):
    img[:, col] = int(col / w * 255)

# Add a bright square
img[30:60, 30:60] = 220
# Add a dark square
img[70:100, 70:100] = 30

plt.figure(figsize=(4, 4))
plt.imshow(img, cmap='gray', vmin=0, vmax=255)
plt.title('Original Test Image')
plt.axis('off')
plt.show()

## 2. Normalization

In [ ]:
norm_minmax = minicv.utils.normalize(img, 'minmax')
norm_zscore = minicv.utils.normalize(img, 'zscore')
norm_uint8 = minicv.utils.normalize(img, 'uint8')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(norm_minmax, cmap='gray'); axes[0].set_title('MinMax [0,1]')
axes[1].imshow(norm_zscore, cmap='gray'); axes[1].set_title('Z-Score')
axes[2].imshow(norm_uint8, cmap='gray'); axes[2].set_title('uint8 [0,255]')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

print(f"MinMax range: [{norm_minmax.min():.2f}, {norm_minmax.max():.2f}]")
print(f"Z-Score mean: {norm_zscore.mean():.4f}, std: {norm_zscore.std():.4f}")

## 3. Filtering

In [ ]:
# Add salt-and-pepper noise
noisy = img.copy().astype(np.float64)
rng = np.random.default_rng(42)
salt = rng.random(img.shape) < 0.02
pepper = rng.random(img.shape) < 0.02
noisy[salt] = 255
noisy[pepper] = 0
noisy = noisy.astype(np.uint8)

mean_result = minicv.filtering.mean_filter(noisy, 3)
gauss_result = minicv.filtering.gaussian_filter(noisy, 5, 1.0)
median_result = minicv.filtering.median_filter(noisy, 3)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(noisy, cmap='gray'); axes[0].set_title('Noisy')
axes[1].imshow(mean_result, cmap='gray'); axes[1].set_title('Mean 3x3')
axes[2].imshow(gauss_result, cmap='gray'); axes[2].set_title('Gaussian 5x5')
axes[3].imshow(median_result, cmap='gray'); axes[3].set_title('Median 3x3')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Thresholding

In [ ]:
thresh_global = minicv.processing.threshold(img, 'global', thresh=128)
thresh_otsu = minicv.processing.threshold(img, 'otsu')
thresh_adaptive = minicv.processing.threshold(img, 'adaptive', block_size=11, adaptive_method='mean', C=5)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original')
axes[1].imshow(thresh_global, cmap='gray'); axes[1].set_title('Global (T=128)')
axes[2].imshow(thresh_otsu, cmap='gray'); axes[2].set_title('Otsu')
axes[3].imshow(thresh_adaptive, cmap='gray'); axes[3].set_title('Adaptive Mean')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Sobel Edge Detection

In [ ]:
gx, gy, mag, direction = minicv.processing.sobel(img)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(gx, cmap='gray'); axes[0].set_title('Gx (Horizontal)')
axes[1].imshow(gy, cmap='gray'); axes[1].set_title('Gy (Vertical)')
axes[2].imshow(mag, cmap='gray'); axes[2].set_title('Magnitude')
axes[3].imshow(direction, cmap='hsv'); axes[3].set_title('Direction')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Bit-Plane Slicing

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(8):
    plane = minicv.processing.bit_plane_slice(img, i)
    ax = axes[i // 4, i % 4]
    ax.imshow(plane, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Bit Plane {i}')
    ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Histogram & Equalization

In [ ]:
hist = minicv.processing.histogram(img)
equalized = minicv.processing.equalize_histogram(img)
hist_eq = minicv.processing.histogram(equalized)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].imshow(img, cmap='gray'); axes[0, 0].set_title('Original')
axes[0, 1].bar(range(256), hist, width=1); axes[0, 1].set_title('Original Histogram')
axes[1, 0].imshow(equalized, cmap='gray'); axes[1, 0].set_title('Equalized')
axes[1, 1].bar(range(256), hist_eq, width=1); axes[1, 1].set_title('Equalized Histogram')
axes[0, 0].axis('off'); axes[1, 0].axis('off')
plt.tight_layout(); plt.show()

## 8. Laplacian Sharpening & Gamma Correction

In [ ]:
sharpened = minicv.processing.laplacian_sharpen(img, strength=2.0)
gamma_bright = minicv.processing.gamma_correction(img, gamma=0.5)
gamma_dark = minicv.processing.gamma_correction(img, gamma=2.0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original')
axes[1].imshow(sharpened, cmap='gray'); axes[1].set_title('Laplacian Sharpened')
axes[2].imshow(gamma_bright, cmap='gray'); axes[2].set_title('Gamma=0.5 (Bright)')
axes[3].imshow(gamma_dark, cmap='gray'); axes[3].set_title('Gamma=2.0 (Dark)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 9. Geometric Transforms

In [ ]:
resized = minicv.transforms.resize(img, (64, 64), method='bilinear')
rotated = minicv.transforms.rotate(img, 30)
translated = minicv.transforms.translate(img, 20, -10)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original (128x128)')
axes[1].imshow(resized, cmap='gray'); axes[1].set_title('Resized (64x64)')
axes[2].imshow(rotated, cmap='gray'); axes[2].set_title('Rotated 30°')
axes[3].imshow(translated, cmap='gray'); axes[3].set_title('Translated (20, -10)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 10. Feature Extraction

In [ ]:
color_hist = minicv.features.color_histogram_descriptor(img, bins=32)
pixel_stats = minicv.features.pixel_statistics_descriptor(img)
hog = minicv.features.hog_descriptor(img, cell_size=16, bins=9)
edge_hist = minicv.features.edge_orientation_histogram(img, bins=36)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].bar(range(len(color_hist)), color_hist); axes[0, 0].set_title(f'Color Histogram ({len(color_hist)}D)')
axes[0, 1].bar(['mean', 'std', 'min', 'max', 'skew', 'kurt'], pixel_stats)
axes[0, 1].set_title('Pixel Statistics'); axes[0, 1].tick_params(axis='x', rotation=45)
axes[1, 0].bar(range(len(hog)), hog); axes[1, 0].set_title(f'HOG ({len(hog)}D)')
axes[1, 1].bar(range(len(edge_hist)), edge_hist); axes[1, 1].set_title(f'Edge Orientation ({len(edge_hist)}D)')
plt.tight_layout(); plt.show()

## 11. Drawing Primitives

In [ ]:
canvas = np.ones((128, 256, 3), dtype=np.uint8) * 240  # light gray canvas

# Draw shapes
minicv.drawing.draw_point(canvas, 20, 20, color=(255, 0, 0), thickness=5)
minicv.drawing.draw_line(canvas, 10, 50, 120, 80, color=(0, 128, 255), thickness=2)
minicv.drawing.draw_rectangle(canvas, 140, 10, 60, 40, color=(0, 200, 0), thickness=2)
minicv.drawing.draw_rectangle(canvas, 140, 60, 60, 40, color=(200, 100, 50), filled=True)
minicv.drawing.draw_polygon(canvas, [(40, 90), (80, 90), (100, 120), (60, 125), (20, 120)],
                             color=(128, 0, 255), thickness=2)

plt.figure(figsize=(8, 4))
plt.imshow(canvas)
plt.title('Drawing Primitives')
plt.axis('off')
plt.show()

## 12. I/O Round-Trip

In [ ]:
import tempfile, os

# Save and reload
with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
    path = f.name

minicv.io.save_image(img, path)
loaded = minicv.io.read_image(path)
os.unlink(path)

print(f'Saved shape: {img.shape}, Loaded shape: {loaded.shape}')
print(f'Loaded dtype: {loaded.dtype}')

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original')
axes[1].imshow(loaded, cmap='gray'); axes[1].set_title('Saved & Reloaded')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---
**End of demo.** All MiniCV functions demonstrated successfully.